# 02 — Retrieval Deneyleri (İP-3)

Tüm sistemleri (5 embedding modeli + BM25) aynı soru setinde yarıştırır: Recall@5/10, nDCG@10, MRR@10, Hit@5, gecikme.

Bu notebook ince bir kabuktur; asıl mantık `scripts/deney_calistir.py` içindedir ve komut satırından da çalışır.

**Ön koşul:** `data/processed/sorular.jsonl` hazır olmalı (notebook 01).

In [ ]:
# Deneyi çalıştır (GPU gerekir; modeller ilk seferde indirilir)
import os
REPO_URL = 'https://github.com/EmreK37/RAG-Tabanli-Turkce-Soru-Cevap-Sistemleri.git'
BRANCH = 'RAG-dev'
REPO_DIR = '/content/RAG-proje'
if not os.path.exists(REPO_DIR):
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git pull origin {BRANCH}
!pip install -q -r requirements.txt
!python scripts/deney_calistir.py --config configs/deney.yaml

In [ ]:
# En güncel özet tabloyu göster
from pathlib import Path
import pandas as pd

klasorler = sorted(Path('results').glob('ana-karsilastirma-*'))
son = klasorler[-1]
ozet = pd.read_csv(son / 'ozet.csv')
print('Deney klasörü:', son)
ozet

In [ ]:
# Eşleştirilmiş bootstrap: en iyi dense model vs BM25 (nDCG@10)
import sys
sys.path.insert(0, '/content/RAG-proje')
import numpy as np
from src.metrics import eslestirilmis_bootstrap_fark

soru = pd.read_csv(son / 'soru-bazinda.csv')
metrik = soru[soru['metrik'] == 'ndcg@10']
pivot = metrik.pivot(index='soru_id', columns='sistem', values='skor')

en_iyi_dense = ozet[ozet['sistem'] != 'bm25'].sort_values('ndcg@10', ascending=False).iloc[0]['sistem']
a = pivot[en_iyi_dense].to_numpy()
b = pivot['bm25'].to_numpy()
sonuc = eslestirilmis_bootstrap_fark(a, b, tekrar=1000)
print(f'{en_iyi_dense} - bm25 | fark: {sonuc["fark"]:.4f} '
      f'GA: [{sonuc["alt"]:.4f}, {sonuc["ust"]:.4f}] anlamlı: {sonuc["anlamli"]}')
print('\nSonuç klasörünü Drive\'a yedeklemeyi unutmayın.')

## Yorum rehberi

- Güven aralığı 0'ı içermiyorsa fark istatistiksel olarak anlamlıdır (H1 için kanıt).
- Soru tipine göre kırılım için `soru-bazinda.csv` içindeki `tip` sütununu kullanın (RQ2).
- TR-MTEB/TR-TEB standart görevleri ayrı çalışmalardır; `datasets` ile indirilip aynı metrik koduyla değerlendirilir.